[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-deepmind/regress-lm/blob/main/colabs/triton_demo.ipynb)

# GPU Kernel Optimization with RegressLM

Fine-tune a pretrained RegressLM checkpoint on Triton GPU kernel code and evaluate its ability to predict kernel execution time — enabling search-based kernel optimization.

Recommended: H100-class GPU or better.

In [ ]:
!pip install -q "regress-lm[extras] @ git+https://github.com/google-deepmind/regress-lm.git"

In [ ]:
import math
import logging
import pickle
import numpy as np
import torch
from regress_lm import core, rlm, tokenizers, vocabs
from regress_lm.pytorch import fine_tuning, optimizers

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Prerequisite: Hugging Face Token

The model uses a gated T5Gemma-2 checkpoint. Before continuing:
1. Accept the license agreement on the [`google/t5gemma-2-270m-270m`](https://huggingface.co/google/t5gemma-2-270m-270m) Hugging Face page.
2. Generate a "Read" token at [`huggingface.co/settings/tokens`](https://huggingface.co/settings/tokens).
3. Add the token to Colab's Secrets tab (key icon on the left) as `HF_TOKEN` and enable "Notebook access".

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

## 1. Load Pretrained Checkpoint

We load a RegressLM checkpoint pretrained on Triton kernel runtime data. The model uses a [T5Gemma-2](https://blog.google/technology/developers/t5gemma-2/) 270M encoder with a 12-layer Transformer decoder.

In [ ]:
CHECKPOINT_URL = 'gs://regress-lm/checkpoints/triton/best_checkpoint.pt'  # TODO: update

decoder_vocab = vocabs.DecoderVocab(
    tokenizers.AddSpecialValues(
        tokenizers.IEEEFloatTokenizer(num_mantissa_digits=5),
        special_value_map={'NAN': math.nan},
    )
)

rlm_wrapper = rlm.RegressLM.from_t5gemma_encoder(
    model_name='google/t5gemma-2-270m-270m',
    decoder_vocab=decoder_vocab,
    max_input_len=8192,
    max_num_objs=1,
    num_decoder_layers=12,
)
model = rlm_wrapper.model
model.to(device)

# Load pretrained weights
import gcsfs
fs = gcsfs.GCSFileSystem()
with fs.open(CHECKPOINT_URL, 'rb') as f:
    state = torch.load(f, map_location='cpu')
model.load_state_dict(state['model_state'])
print(f'Loaded checkpoint. Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

## 2. Prepare Triton Kernel Data

Each example is a Triton kernel implementation paired with its measured execution time (lower = faster).

In [ ]:
# --- Demo data ---
# In practice, replace with real Triton kernel data.
# Format: core.Example(x='<problem_id>\n\n<kernel_code>', y=<runtime_ms>)

def make_triton_demo_data(n_kernels=500, seed=42):
    """Creates synthetic Triton kernel examples for demonstration."""
    rng = np.random.default_rng(seed)
    examples = []
    problem_name = 'matmul_256x256'
    for i in range(n_kernels):
        block_m = rng.choice([32, 64, 128, 256])
        block_n = rng.choice([32, 64, 128, 256])
        block_k = rng.choice([16, 32, 64])
        num_warps = rng.choice([2, 4, 8])
        code = f'import triton\nimport triton.language as tl\n\n@triton.jit\ndef matmul_kernel(\n    a_ptr, b_ptr, c_ptr, M, N, K,\n    BLOCK_M: tl.constexpr = {block_m},\n    BLOCK_N: tl.constexpr = {block_n},\n    BLOCK_K: tl.constexpr = {block_k},\n):\n    pid_m = tl.program_id(0)\n    pid_n = tl.program_id(1)\n    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)\n    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)\n    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)\n    for k in range(0, K, BLOCK_K):\n        a = tl.load(a_ptr + offs_m[:, None] * K + k + tl.arange(0, BLOCK_K))\n        b = tl.load(b_ptr + (k + tl.arange(0, BLOCK_K)[:, None]) * N + offs_n)\n        acc += tl.dot(a, b)\n    tl.store(c_ptr + offs_m[:, None] * N + offs_n, acc)\n# num_warps={num_warps}'
        time_ms = 0.5 + abs(block_m - 128) * 0.01 + abs(block_n - 128) * 0.01 + rng.normal(0, 0.05)
        examples.append(core.Example(x=f'{problem_name}\n\n{code}', y=float(max(0.1, time_ms))))
    return examples

all_examples = make_triton_demo_data(n_kernels=500)
print(f'Total kernels: {len(all_examples)}')
print(f'Runtime range: [{min(e.y for e in all_examples):.3f}, {max(e.y for e in all_examples):.3f}] ms')
print(f'\nExample kernel (first 300 chars):\n{all_examples[0].x[:300]}')

## 3. Fine-Tuning

We fine-tune all model parameters on 256 kernels with a held-out validation set for early stopping.

In [ ]:
from regress_lm.pytorch import fine_tuning, optimizers

SEED = 42
N_TRAIN = 256
N_VAL = 64

# Shuffle and split into train / val / test
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(all_examples))
train_examples = [all_examples[i] for i in perm[:N_TRAIN]]
val_examples = [all_examples[i] for i in perm[N_TRAIN:N_TRAIN + N_VAL]]
test_examples = [all_examples[i] for i in perm[N_TRAIN + N_VAL:]]

print(f'Train: {len(train_examples)}, Val: {len(val_examples)}, Test: {len(test_examples)}')

def optimizer_factory(named_params):
    trainable = [(n, p) for n, p in named_params if p.requires_grad]
    return optimizers.muon_adamw(trainable, lr=1e-5)

fine_tuner = fine_tuning.PyTorchFineTuner(
    model,
    optimizer_factory=optimizer_factory,
    max_epochs=10_000,
    batch_size=16,
    batch_size_per_device=4,  # fits on H100 80GB; use 8 for B200
    patience=10,
    use_lora=False,
)

fine_tuner.fine_tune(
    examples=train_examples,
    validation_examples=val_examples,
    seed=SEED,
)
print('Fine-tuning complete!')

## 4. Inference & Ranking Evaluation

For kernel optimization, the key metric is **ranking quality**: can the model identify the fastest kernels without running them?

In [ ]:
NUM_SAMPLES = 128
model.eval()

all_preds = []
with torch.inference_mode():
    for ex in test_examples:
        samples = rlm_wrapper.sample([core.ExampleInput(x=ex.x)], NUM_SAMPLES)
        all_preds.append(samples[0])  # (num_samples,)

preds = np.stack(all_preds)  # (N, num_samples)
y_true = np.array([ex.y for ex in test_examples])
y_pred = np.nanmedian(preds, axis=1)

print(f'Predictions shape: {preds.shape}')
print(f'NaN rate: {np.isnan(preds).mean():.1%}')

In [ ]:
from scipy import stats

valid = np.isfinite(y_true) & np.isfinite(y_pred)
y_t, y_p = y_true[valid], y_pred[valid]

print('=== Regression Metrics ===')
print(f'MSE:      {np.mean((y_t - y_p) ** 2):.6f}')
print(f'MAE:      {np.mean(np.abs(y_t - y_p)):.6f}')
print(f'Pearson:  {stats.pearsonr(y_t, y_p)[0]:.4f}')
print(f'Spearman: {stats.spearmanr(y_t, y_p)[0]:.4f}')

print('\n=== Ranking Metrics (lower runtime = better) ===')
for k in [1, 5, 10]:
    true_topk = set(np.argsort(y_true)[:k])
    pred_topk = set(np.argsort(y_pred)[:k])
    recall = len(true_topk & pred_topk) / k
    model_best = y_true[np.argsort(y_pred)[:k]].min()
    oracle_best = y_true.min()
    print(f'Top-{k}: Recall={recall:.0%}, Best={model_best:.4f}ms (Oracle={oracle_best:.4f}ms)')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# True vs Predicted
axes[0].scatter(y_true, y_pred, alpha=0.6, s=20, c='steelblue')
min_val, max_val = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5)
axes[0].set_xlabel('True Runtime [ms]'); axes[0].set_ylabel('Predicted [ms]')
axes[0].set_title(f'Runtime Prediction')

# Search efficiency
model_order = np.argsort(y_pred)
random_order = np.arange(len(y_true)); np.random.shuffle(random_order)
def cumulative_best(order, y): return np.minimum.accumulate(y[order])
axes[1].plot(cumulative_best(model_order, y_true), label='Model-guided', lw=2)
axes[1].plot(cumulative_best(random_order, y_true), label='Random', lw=2, ls='--')
axes[1].axhline(y_true.min(), color='green', ls=':', lw=1, label='Oracle')
axes[1].set_xlabel('Kernels Tried'); axes[1].set_ylabel('Best Runtime [ms]')
axes[1].set_title('Search Efficiency'); axes[1].legend()

# Uncertainty
sorted_idx = np.argsort(y_true)
axes[2].fill_between(range(len(y_true)),
    np.nanpercentile(preds[sorted_idx], 25, axis=1),
    np.nanpercentile(preds[sorted_idx], 75, axis=1),
    alpha=0.3, color='steelblue', label='25-75th pct')
axes[2].plot(y_true[sorted_idx], 'k-', lw=1.5, label='True')
axes[2].plot(np.nanmedian(preds[sorted_idx], axis=1), 'r-', lw=1, label='Predicted')
axes[2].set_xlabel('Kernel (sorted)'); axes[2].set_ylabel('Runtime [ms]')
axes[2].set_title('Predictions + Uncertainty'); axes[2].legend()

plt.tight_layout()
plt.show()